In [1]:
"""
train_model.py
──────────────
Run this ONCE to train the RandomForest model and save model.pkl.

Usage:
    python train_model.py

Output:
    model.pkl  ← load this in app.py
"""

import pandas as pd
import numpy as np
import pickle
import warnings
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────
DATA_PATH  = "QCFE9_Perfect_Discount_Statergy.csv"
MODEL_PATH = "model.pkl"

# ─────────────────────────────────────────────────────────────
# 1. LOAD DATA
# ─────────────────────────────────────────────────────────────
print("📂 Loading dataset…")
df = pd.read_csv(DATA_PATH)
df["Order_date"] = pd.to_datetime(df["Order_date"])
df["Month"]      = df["Order_date"].dt.month
df["DayOfYear"]  = df["Order_date"].dt.dayofyear
print(f"   ✅ Loaded {len(df):,} rows × {df.shape[1]} columns")

# ─────────────────────────────────────────────────────────────
# 2. FEATURE ENGINEERING
# ─────────────────────────────────────────────────────────────
print("\n⚙️  Building features…")
agg = df.groupby(["City", "Order_date", "Product_Category"]).agg(
    items_sum      =("Items_Count",               "sum"),
    avg_demand     =("Avg_Daily_Demand",           "mean"),
    pref_score     =("Customer_Preference_Score",  "mean"),
    peak_intensity =("Peak_Intensity",             "mean"),
    is_festival    =("Is_Festival",                "max"),
    is_weekend     =("Is_Weekend",                 "max"),
    order_count    =("Order_ID",                   "count"),
    season         =("Season",                     "first"),
    weather        =("Weather",                    "first"),
    month          =("Month",                      "first"),
    day_of_year    =("DayOfYear",                  "first"),
).reset_index()

# Category share % per city+date
agg["total_items"] = agg.groupby(["City", "Order_date"])["items_sum"].transform("sum")
agg["share_pct"]   = (agg["items_sum"] / agg["total_items"] * 100).round(2)

# Demand label: High ≥ 20%  |  Low < 12%  |  Medium in between
agg["demand_label"] = agg["share_pct"].apply(
    lambda x: "High" if x >= 20 else ("Low" if x < 12 else "Medium")
)

print(f"   ✅ Aggregated to {len(agg):,} city-date-category rows")
print(f"\n   Label distribution:")
for label, count in agg["demand_label"].value_counts().items():
    print(f"      {label:8s}: {count:,}")

# ─────────────────────────────────────────────────────────────
# 3. ENCODE CATEGORICALS
# ─────────────────────────────────────────────────────────────
print("\n🔠 Encoding categorical columns…")
le_city    = LabelEncoder().fit(agg["City"])
le_cat     = LabelEncoder().fit(agg["Product_Category"])
le_season  = LabelEncoder().fit(agg["season"])
le_weather = LabelEncoder().fit(agg["weather"])

agg["city_enc"]    = le_city.transform(agg["City"])
agg["cat_enc"]     = le_cat.transform(agg["Product_Category"])
agg["season_enc"]  = le_season.transform(agg["season"])
agg["weather_enc"] = le_weather.transform(agg["weather"])

FEATURES = [
    "city_enc", "cat_enc", "avg_demand", "pref_score",
    "peak_intensity", "is_festival", "is_weekend",
    "order_count", "season_enc", "weather_enc",
    "month", "day_of_year",
]

# ─────────────────────────────────────────────────────────────
# 4. TRAIN / TEST SPLIT
# ─────────────────────────────────────────────────────────────
X = agg[FEATURES]
y = agg["demand_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\n📊 Train: {len(X_train):,}  |  Test: {len(X_test):,}")

# ─────────────────────────────────────────────────────────────
# 5. TRAIN MODEL
# ─────────────────────────────────────────────────────────────
print("\n🤖 Training RandomForest (150 trees, depth 12)…")
model = RandomForestClassifier(
    n_estimators=150,
    max_depth=12,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1,
)
model.fit(X_train, y_train)

# ─────────────────────────────────────────────────────────────
# 6. EVALUATE
# ─────────────────────────────────────────────────────────────
y_pred   = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"\n📈 Test Accuracy: {accuracy * 100:.2f}%")
print("\n" + classification_report(y_test, y_pred))

# ─────────────────────────────────────────────────────────────
# 7. SAVE model.pkl
# ─────────────────────────────────────────────────────────────
bundle = {
    "model":      model,
    "agg":        agg,          # historical lookup table for share_pct
    "le_city":    le_city,
    "le_cat":     le_cat,
    "le_season":  le_season,
    "le_weather": le_weather,
    "features":   FEATURES,
    "accuracy":   round(accuracy * 100, 2),
}

with open(MODEL_PATH, "wb") as f:
    pickle.dump(bundle, f)

print(f"✅ Model saved → {MODEL_PATH}")
print(f"   Cities  : {le_city.classes_.tolist()}")
print(f"   Categories: {le_cat.classes_.tolist()}")

📂 Loading dataset…
   ✅ Loaded 985,095 rows × 62 columns

⚙️  Building features…
   ✅ Aggregated to 8,371 city-date-category rows

   Label distribution:
      Low     : 3,592
      Medium  : 3,131
      High    : 1,648

🔠 Encoding categorical columns…

📊 Train: 6,696  |  Test: 1,675

🤖 Training RandomForest (150 trees, depth 12)…

📈 Test Accuracy: 79.88%

              precision    recall  f1-score   support

        High       0.59      0.59      0.59       330
         Low       0.93      0.96      0.95       719
      Medium       0.75      0.72      0.73       626

    accuracy                           0.80      1675
   macro avg       0.76      0.76      0.76      1675
weighted avg       0.80      0.80      0.80      1675

✅ Model saved → model.pkl
   Cities  : ['Amritsar', 'Bengluru', 'Chennai', 'Delhi', 'Gurgaon', 'Haridwar', 'Hyderabad', 'Jaipur', 'Kolkata', 'Mumbai', 'Noida', 'Pune']
   Categories: ['Beverages', 'Dairy', 'Fruits & Vegetables', 'Groceries', 'Household', 'Pers